In [1]:
from pathlib import Path
import sys
import os

PROJECT_ROOT = Path.cwd().parent

import pandas as pd

import src.constants as Con

In [ ]:
#TOTAL RT AS FEATURE

#look at the probabilities, actually

In [2]:
from src.data_paths import (
    EXPERIMENT_TEXT_COMPLETED_PATH,

    DATA_DIR,
    EXPERIMENT_DIR
)

In [3]:
texts = pd.read_csv(EXPERIMENT_TEXT_COMPLETED_PATH, encoding="utf-8")
additional_reqs = pd.read_csv(DATA_DIR / "all_participants_with_practice.csv", encoding="utf-8")

In [4]:
extra_cols = ['onestopqa_question_id', Con.ARTICLE_COLUMN, Con.PARAGRAPH_COLUMN, Con.DIFFICULTY_COLUMN, Con.BATCH_COLUMN]


reqs_lookup = (
    additional_reqs[['text_id_with_q'] + extra_cols]
    .drop_duplicates(subset='text_id_with_q')
)

texts_with_ids = texts.merge(reqs_lookup, on='text_id_with_q', how='left')
texts_with_ids

,text_id_with_q,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,paragraph_id,difficulty_level,article_batch
0,1_0_Adv_1_0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,Adv,1
1,1_0_Adv_2_0,â€œThere will be just enough water if the prop...,Which factor led to the increase in the price ...,Unfavorable weather conditions in different lo...,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,Global warming,Bad weather in different locations on the planet,0,0,2,Adv,1
2,1_10_Adv_1_0,"Agios Efstratios is so remote, so forgotten by...",What event dramatically reduced the quality of...,The financial crisis that occurred in the country,A storm in the northern Aegean sea that devast...,The removal of ATM machines from the island,The installment of a new government,The national economic crisis,0,10,1,Adv,1
3,1_10_Adv_1_1,"Agios Efstratios is so remote, so forgotten by...",What was true about tourism in Agios Efstratio...,A small number of tourists visited Agios Efstr...,Tourists overcrowded Agios Efstratios,Agios Efstratios was so remote that tourists w...,Many tourism agencies gave awards to Agios Efs...,A relatively small group of tourists visited t...,1,10,1,Adv,1
4,1_10_Adv_1_2,"Agios Efstratios is so remote, so forgotten by...",Are there many hotels in Agios Efstratios?,"No, as there are only a small number of rooms ...","No, the beaches are empty of hotels","Yes, they are all in remote locations","Yes, the island has several 5-star resorts","No, the accommodation is limited to a small nu...",2,10,1,Adv,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
973,3_9_Ele_3_1,SOHAY also has classes for parents and employe...,What do Zhorna's brothers do?,They are both employed so that they can make m...,"One works at a restaurant and a club, and the ...",They try their best to not drop out of school ...,They help Zhorna with her homework after school,Both brothers have jobs to help their parents ...,1,9,3,Ele,3
974,3_9_Ele_3_2,SOHAY also has classes for parents and employe...,Why do Zhorna's brothers both work instead of ...,Their family is reliant on their salaries to m...,They prefer helping their parents,They found it too difficult to adapt to school,The nearest school is too far to commute to daily,Their family depends on the money they earn,2,9,3,Ele,3
975,3_9_Ele_4_0,"In 2015, SOHAY helped 1,540 children to leave ...",What statistic does UNICEF provide about child...,1.3 million children are employed in unsafe wo...,4.7 million children are unable to attend school,"1,540 children were able to leave risky work i...",The children labor rates in Bangladesh are hig...,1.3 million children work in hazardous industries,1,9,4,Ele,3
976,3_9_Ele_4_1,"In 2015, SOHAY helped 1,540 children to leave ...",What did SOHAY do in 2015?,"Assisted 2,125 children in attending school in...","Provided funds for 1,540 children in danger of...",Collaborated with UNICEF to reduce child labor,"Provided free classes for 3,000 children","Assisted 2,125 children in going to school rat...",0,9,4,Ele,3


In [5]:
adv = texts_with_ids[texts_with_ids['difficulty_level'] == 'Adv']
ele = texts_with_ids[texts_with_ids['difficulty_level'] == 'Ele']
adv.to_csv(EXPERIMENT_DIR / "adv_texts_with_ids.csv", index=False, encoding="utf-8")
ele.to_csv(EXPERIMENT_DIR / "ele_texts_with_ids.csv", index=False, encoding="utf-8")

In [6]:
def create_article_regime_design(
    df: pd.DataFrame,
    practice_article_id: int = 0,
    batch_col: str = "article_batch",
    article_col: str = "article_id",
    regime_partial: str = "partial",
    regime_full: str = "full",
    regime_none: str = "no_knowledge",
    display_tables: bool = True,
    include_practice_in_display: bool = False,
):
    """
    Create article-level Latin-square regime assignments and merge them
    back onto the paragraph/question-level dataframe.

    Design logic:
    - Each batch contains one practice article, default article_id == 0.
    - Each batch contains 10 experimental articles.
    - Experimental articles 1-9 are split into 3 groups of 3.
    - These 3 groups rotate through:
        partial, full, no_knowledge.
    - The 10th experimental article alternates between:
        full and no_knowledge.
    - The practice article is included once in each regime.
    - No block/article presentation order is added here.

    Returns:
    - adv_with_regimes: original dataframe expanded by article-regime list assignments
    - article_design: article-level assignment table
    - latin_square_table: readable summary table
    """

    def build_one_batch(batch_id):
        batch_df = df[df[batch_col] == batch_id].copy()

        article_ids = sorted(batch_df[article_col].unique())

        if practice_article_id not in article_ids:
            raise ValueError(
                f"Batch {batch_id} does not contain practice article "
                f"{practice_article_id}."
            )

        experimental_ids = [
            article_id
            for article_id in article_ids
            if article_id != practice_article_id
        ]

        if len(experimental_ids) != 10:
            raise ValueError(
                f"Batch {batch_id} has {len(experimental_ids)} experimental articles, "
                f"but expected exactly 10. Found: {experimental_ids}"
            )

        t1, t2, t3, t4, t5, t6, t7, t8, t9, t10 = experimental_ids

        group_1 = [t1, t2, t3]
        group_2 = [t4, t5, t6]
        group_3 = [t7, t8, t9]
        extra = t10

        core_rotations = [
            {
                regime_partial: group_1,
                regime_full: group_2,
                regime_none: group_3,
            },
            {
                regime_partial: group_2,
                regime_full: group_3,
                regime_none: group_1,
            },
            {
                regime_partial: group_3,
                regime_full: group_1,
                regime_none: group_2,
            },
        ]

        extra_variants = {
            "a": regime_full,
            "b": regime_none,
        }

        rows = []

        for rotation_index, rotation in enumerate(core_rotations, start=1):
            for variant_label, extra_regime in extra_variants.items():

                list_version = f"{rotation_index}{variant_label}"
                list_id = f"B{batch_id}_{list_version}"

                # Practice article appears once in each regime
                for regime in [regime_partial, regime_full, regime_none]:
                    rows.append({
                        batch_col: batch_id,
                        "list_id": list_id,
                        "list_version": list_version,
                        "article_regime": regime,
                        article_col: practice_article_id,
                        "is_practice": True,
                    })

                # Experimental articles
                for regime, articles in rotation.items():
                    assigned_articles = list(articles)

                    if regime == extra_regime:
                        assigned_articles.append(extra)

                    for article_id in assigned_articles:
                        rows.append({
                            batch_col: batch_id,
                            "list_id": list_id,
                            "list_version": list_version,
                            "article_regime": regime,
                            article_col: article_id,
                            "is_practice": False,
                        })

        return pd.DataFrame(rows)

    # Build article-level design for all batches
    article_design = pd.concat(
        [
            build_one_batch(batch_id)
            for batch_id in sorted(df[batch_col].unique())
        ],
        ignore_index=True,
    )

    # Merge design back onto full paragraph/question dataframe
    adv_with_regimes = df.merge(
        article_design,
        on=[batch_col, article_col],
        how="inner",
    )

    # Create readable Latin-square table
    display_design = article_design.copy()

    if not include_practice_in_display:
        display_design = display_design[~display_design["is_practice"]].copy()

    latin_square_table = (
        display_design
        .groupby([batch_col, "list_version", "article_regime"])[article_col]
        .apply(lambda ids: ", ".join(map(str, sorted(ids))))
        .reset_index()
        .pivot_table(
            index=[batch_col, "list_version"],
            columns="article_regime",
            values=article_col,
            aggfunc="first",
        )
        .reset_index()
    )

    wanted_columns = [
        batch_col,
        "list_version",
        regime_partial,
        regime_full,
        regime_none,
    ]

    latin_square_table = latin_square_table[wanted_columns]

    if display_tables:
        for batch_id in sorted(latin_square_table[batch_col].unique()):
            print(f"\nBatch {batch_id}")
            print("-" * 80)

            batch_table = (
                latin_square_table[latin_square_table[batch_col] == batch_id]
                .drop(columns=[batch_col])
                .reset_index(drop=True)
            )

            display(batch_table)

        print("\nSummary")
        print("-" * 80)
        print("Original rows:", len(df))
        print("Rows after adding regime lists:", len(adv_with_regimes))
        print("Number of article-regime lists:", adv_with_regimes["list_id"].nunique())

    return adv_with_regimes, article_design, latin_square_table

In [7]:
adv_with_regimes, article_design, latin_square_table = create_article_regime_design(
    adv,
    display_tables=True,
    include_practice_in_display=False,
)


Batch 1
--------------------------------------------------------------------------------


article_regime,list_version,partial,full,no_knowledge
0,1a,"1, 2, 3","4, 5, 6, 10","7, 8, 9"
1,1b,"1, 2, 3","4, 5, 6","7, 8, 9, 10"
2,2a,"4, 5, 6","7, 8, 9, 10","1, 2, 3"
3,2b,"4, 5, 6","7, 8, 9","1, 2, 3, 10"
4,3a,"7, 8, 9","1, 2, 3, 10","4, 5, 6"
5,3b,"7, 8, 9","1, 2, 3","4, 5, 6, 10"



Batch 2
--------------------------------------------------------------------------------


article_regime,list_version,partial,full,no_knowledge
0,1a,"1, 2, 3","4, 5, 6, 10","7, 8, 9"
1,1b,"1, 2, 3","4, 5, 6","7, 8, 9, 10"
2,2a,"4, 5, 6","7, 8, 9, 10","1, 2, 3"
3,2b,"4, 5, 6","7, 8, 9","1, 2, 3, 10"
4,3a,"7, 8, 9","1, 2, 3, 10","4, 5, 6"
5,3b,"7, 8, 9","1, 2, 3","4, 5, 6, 10"



Batch 3
--------------------------------------------------------------------------------


article_regime,list_version,partial,full,no_knowledge
0,1a,"1, 2, 3","4, 5, 6, 10","7, 8, 9"
1,1b,"1, 2, 3","4, 5, 6","7, 8, 9, 10"
2,2a,"4, 5, 6","7, 8, 9, 10","1, 2, 3"
3,2b,"4, 5, 6","7, 8, 9","1, 2, 3, 10"
4,3a,"7, 8, 9","1, 2, 3, 10","4, 5, 6"
5,3b,"7, 8, 9","1, 2, 3","4, 5, 6, 10"



Summary
--------------------------------------------------------------------------------
Original rows: 492
Rows after adding regime lists: 3024
Number of article-regime lists: 18


In [8]:
def create_question_rotated_lists(
    adv_with_regimes_base: pd.DataFrame,
    question_id_col: str = "onestopqa_question_id",
    batch_col: str = "article_batch",
    article_col: str = "article_id",
    paragraph_col: str = "paragraph_id",
    practice_col: str = "is_practice",
    list_col: str = "list_id",
) -> pd.DataFrame:
    """
    Create 3 question-list versions per article-regime list.

    Experimental paragraphs:
        selected question = (paragraph_index + question_list_version) % 3

    Practice paragraphs:
        always selected question 0

    Important:
    - question_list_version is NOT the actual question ID.
    - onestopqa_question_id is the actual question ID: 0, 1, or 2.
    """

    df_base = adv_with_regimes_base.copy()

    # Make function safe to rerun
    cols_to_remove = [
        "paragraph_index_in_batch",
        "question_list_version",
        "selected_onestopqa_question_id",
        "final_list_id",
    ]

    df_base = df_base.drop(columns=cols_to_remove, errors="ignore")

    # --------------------------------------------------
    # 1. Build paragraph lookup
    # --------------------------------------------------
    # One paragraph is uniquely identified by:
    # article_batch + article_id + paragraph_id

    paragraph_lookup = (
        df_base[[batch_col, article_col, paragraph_col, practice_col]]
        .drop_duplicates()
        .sort_values([batch_col, article_col, paragraph_col])
        .copy()
    )

    # Create an index only for experimental paragraphs.
    # Practice paragraphs do not need one.
    paragraph_lookup["paragraph_index_in_batch"] = pd.NA

    experimental_mask = ~paragraph_lookup[practice_col]

    paragraph_lookup.loc[experimental_mask, "paragraph_index_in_batch"] = (
        paragraph_lookup.loc[experimental_mask]
        .groupby(batch_col)
        .cumcount()
    )

    paragraph_lookup = paragraph_lookup[
        [batch_col, article_col, paragraph_col, "paragraph_index_in_batch"]
    ]

    df_base = df_base.merge(
        paragraph_lookup,
        on=[batch_col, article_col, paragraph_col],
        how="left",
    )

    # --------------------------------------------------
    # 2. Add three question-list versions
    # --------------------------------------------------

    question_list_versions = pd.DataFrame({
        "question_list_version": [0, 1, 2]
    })

    df_q = df_base.merge(question_list_versions, how="cross")

    # --------------------------------------------------
    # 3. Select question
    # --------------------------------------------------

    # Start empty
    df_q["selected_onestopqa_question_id"] = pd.NA

    # Experimental trials: rotate questions
    experimental_rows = ~df_q[practice_col]

    df_q.loc[experimental_rows, "selected_onestopqa_question_id"] = (
        df_q.loc[experimental_rows, "paragraph_index_in_batch"].astype(int)
        + df_q.loc[experimental_rows, "question_list_version"].astype(int)
    ) % 3

    # Practice trials: always use actual question ID 0
    df_q.loc[df_q[practice_col], "selected_onestopqa_question_id"] = 0

    df_q["selected_onestopqa_question_id"] = (
        df_q["selected_onestopqa_question_id"].astype(int)
    )

    # Keep only the row where the actual question ID matches the selected one
    df_final = df_q[
        df_q[question_id_col] == df_q["selected_onestopqa_question_id"]
    ].copy()

    # --------------------------------------------------
    # 4. Create final list ID
    # --------------------------------------------------

    df_final["final_list_id"] = (
        df_final[list_col].astype(str)
        + "_Q"
        + df_final["question_list_version"].astype(str)
    )

    # --------------------------------------------------
    # 5. Create numeric list number
    # --------------------------------------------------

    list_id_order = sorted(df_final["final_list_id"].unique())

    list_num_map = {
        final_list_id: i
        for i, final_list_id in enumerate(list_id_order)
    }

    df_final["list_num"] = df_final["final_list_id"].map(list_num_map)

    return df_final.reset_index(drop=True)

In [9]:
adv_final_lists = create_question_rotated_lists(adv_with_regimes)

In [10]:
adv_final_lists[adv_final_lists["final_list_id"] == 'B1_1b_Q0']

,text_id_with_q,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,...,article_batch,list_id,list_version,article_regime,is_practice,paragraph_index_in_batch,question_list_version,selected_onestopqa_question_id,final_list_id,list_num
9,1_0_Adv_1_0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,...,1,B1_1b,1b,partial,True,<NA>,0,0,B1_1b_Q0,3
12,1_0_Adv_1_0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,...,1,B1_1b,1b,full,True,<NA>,0,0,B1_1b_Q0,3
15,1_0_Adv_1_0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,...,1,B1_1b,1b,no_knowledge,True,<NA>,0,0,B1_1b_Q0,3
63,1_0_Adv_2_0,â€œThere will be just enough water if the prop...,Which factor led to the increase in the price ...,Unfavorable weather conditions in different lo...,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,Global warming,Bad weather in different locations on the planet,0,0,...,1,B1_1b,1b,partial,True,<NA>,0,0,B1_1b_Q0,3
66,1_0_Adv_2_0,â€œThere will be just enough water if the prop...,Which factor led to the increase in the price ...,Unfavorable weather conditions in different lo...,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,Global warming,Bad weather in different locations on the planet,0,0,...,1,B1_1b,1b,full,True,<NA>,0,0,B1_1b_Q0,3
69,1_0_Adv_2_0,â€œThere will be just enough water if the prop...,Which factor led to the increase in the price ...,Unfavorable weather conditions in different lo...,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,Global warming,Bad weather in different locations on the planet,0,0,...,1,B1_1b,1b,no_knowledge,True,<NA>,0,0,B1_1b_Q0,3
121,1_10_Adv_1_2,"Agios Efstratios is so remote, so forgotten by...",Are there many hotels in Agios Efstratios?,"No, as there are only a small number of rooms ...","No, the beaches are empty of hotels","Yes, they are all in remote locations","Yes, the island has several 5-star resorts","No, the accommodation is limited to a small nu...",2,10,...,1,B1_1b,1b,no_knowledge,False,47,0,2,B1_1b_Q0,3
133,1_10_Adv_2_1,"But, because the island still runs on cash, th...",How has the closure of Greek banks affected th...,They have to make long trips to another island...,They have to go to the biggest Greek island fo...,Some have moved to Athens to earn more money,All the bank workers on the island lost their ...,They have to journey to another island to get ...,0,10,...,1,B1_1b,1b,no_knowledge,False,48,0,0,B1_1b_Q0,3
151,1_10_Adv_3_1,Kakali has badgered the government and a major...,What is the upcoming â€œbigger crisisâ€?,Islands will have to pay additional taxes,The government will end taxes for tourists vis...,The removal of ATMs from the island,The Greek government will shut down in the summer,The islands will lose their long-standing tax ...,1,10,...,1,B1_1b,1b,no_knowledge,False,49,0,1,B1_1b_Q0,3
175,1_10_Adv_4_2,Created to help island communities survive whe...,Why does Provatas Costas mention milk and brea...,To 

In [11]:
adv_final_lists
cols_to_drop = [
    "text_id_with_q",
    "difficulty_level",
    'paragraph_index_in_batch',
    'list_id'
]

df_final_lists = adv_final_lists.drop(columns=cols_to_drop)


In [12]:
df_final_lists.to_csv(EXPERIMENT_DIR / 'exp_design.csv', index=False, encoding="utf-8")

In [13]:
df_final_lists = pd.read_csv(EXPERIMENT_DIR / 'exp_design.csv', encoding="utf-8")

In [14]:
from src.data_paths import (
    N1_BASE_PATH,
    N2_BASE_PATH,
    N3_BASE_PATH,
)

In [15]:
n1_base = pd.read_csv(N1_BASE_PATH, encoding="utf-8")
n2_base = pd.read_csv(N2_BASE_PATH, encoding="utf-8")
n3_base = pd.read_csv(N3_BASE_PATH, encoding="utf-8")

In [16]:
n1_adv = n1_base[(n1_base["level"] == "Adv") & (n1_base['reread'] == 0)]
n2_adv = n2_base[(n2_base["level"] == "Adv") & (n2_base['reread'] == 0)]
n3_adv = n3_base[(n3_base["level"] == "Adv") & (n3_base['reread'] == 0)]

cols = ['batch',"article_id", 'article_title']

n1_adv = n1_adv[cols]
n2_adv = n2_adv[cols]
n3_adv = n3_adv[cols]

n_adv_all = pd.concat(
    [n1_adv, n2_adv, n3_adv],
    ignore_index=True
).drop_duplicates(subset=['batch',"article_id"])

In [17]:
joined = df_final_lists.merge(
    n_adv_all,
    left_on=['article_batch',"article_id"],
    right_on=['batch',"article_id"],
    how="left",
)


In [18]:
joined

,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,paragraph_id,article_batch,list_version,article_regime,is_practice,question_list_version,selected_onestopqa_question_id,final_list_id,list_num,batch,article_title
0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,1,1a,partial,True,0,0,B1_1a_Q0,0,1,Food Shortages Could Force World into Vegetari...
1,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,1,1a,partial,True,1,0,B1_1a_Q1,1,1,Food Shortages Could Force World into Vegetari...
2,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,1,1a,partial,True,2,0,B1_1a_Q2,2,1,Food Shortages Could Force World into Vegetari...
3,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,1,1a,full,True,0,0,B1_1a_Q0,0,1,Food Shortages Could Force World into Vegetari...
4,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,1,1a,full,True,1,0,B1_1a_Q1,1,1,Food Shortages Could Force World into Vegetari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,3,1b,no_knowledge,False,0,2,B3_1b_Q0,39,3,Bangladeshi Organization Delivers a Lesson on ...
3236,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,3,2a,full,False,0,2,B3_2a_Q0,42,3,Bangladeshi Organization Delivers a Lesson on ...
3237,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,3,2b,full,False,0,2,B3_2b_Q0,45,3,Bangladeshi Organization Delivers a Lesson on ...
3238,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,3,3a,partial,False,0,2,B3_3a_Q0,48,3,Bangladeshi Organization Delivers a Lesson on ...


In [19]:
joined.to_csv(EXPERIMENT_DIR / 'exp_design.csv', index=False, encoding="utf-8")

In [20]:
df = pd.read_csv(EXPERIMENT_DIR / 'exp_design.csv', encoding="utf-8")


In [21]:
import numpy as np
import pandas as pd

answer_cols = ["answer_A", "answer_B", "answer_C", "answer_D"]

# 0 = answer_A, 1 = answer_B, 2 = answer_C, 3 = answer_D
rng = np.random.default_rng(seed=42) 

def randomize_answers(row):
    order = rng.permutation(4)  # e.g. [1, 0, 3, 2]

    # answers shown on screen in randomized order
    reordered_answers = {
        f"answer_{screen_pos}": row[answer_cols[original_answer_idx]]
        for screen_pos, original_answer_idx in enumerate(order)
    }

    # keys: where each original answer moved to
    # a_key = position of answer_A, b_key = position of answer_B, etc.
    inverse_order = np.empty(4, dtype=int)
    inverse_order[order] = np.arange(4)

    keys = {
        "a_key": inverse_order[0],
        "b_key": inverse_order[1],
        "c_key": inverse_order[2],
        "d_key": inverse_order[3],
    }

    return pd.Series({
        **reordered_answers,
        "correct_answer": inverse_order[0],   # where answer_A moved
        **keys,
        "answers_order": [int(x) for x in order],
    })

df_with_randomized_answers = df.join(df.apply(randomize_answers, axis=1))

In [22]:
df_with_randomized_answers

,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,paragraph_id,...,answer_0,answer_1,answer_2,answer_3,correct_answer,a_key,b_key,c_key,d_key,answers_order
0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,Obesity rates around the world will rise,"By 2050, animal-based protein consumption will...","By 2050, nine billion people will not have eno...",There will not be sufficient water to grow eno...,3,3,2,1,0,"[3, 2, 1, 0]"
1,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,Obesity rates around the world will rise,"By 2050, animal-based protein consumption will...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...",2,2,3,1,0,"[3, 2, 0, 1]"
2,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,There will not be sufficient water to grow eno...,Obesity rates around the world will rise,"By 2050, animal-based protein consumption will...","By 2050, nine billion people will not have eno...",0,0,3,2,1,"[0, 3, 2, 1]"
3,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,"By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...",2,2,3,0,1,"[2, 3, 0, 1]"
4,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,"By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...",2,2,3,0,1,"[2, 3, 0, 1]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3235,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,...,1.3 million,"2,125","1,540","10,000",2,2,1,0,3,"[2, 1, 0, 3]"
3236,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,...,"10,000",1.3 million,"2,125","1,540",3,3,2,1,0,"[3, 2, 1, 0]"
3237,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 million,"10,000","1,540",2,9,4,...,1.3 million,"10,000","2,125","1,540",3,3,2,0,1,"[2, 3, 1, 0]"
3238,"In 2015, SOHAY helped 1,540 children to leave ...",How many children did SOHAY help leave work in...,"1,540","2,125",1.3 mill

In [23]:
df_with_randomized_answers["is_experimental"] = (~df_with_randomized_answers["is_practice"]).astype(int)

In [24]:
df_with_randomized_answers.to_csv(EXPERIMENT_DIR / 'exp_design_randomized_answers.csv', index=False, encoding="utf-8")

In [25]:
batch1 = df_with_randomized_answers[df_with_randomized_answers['article_batch'] == 1]
batch2 = df_with_randomized_answers[df_with_randomized_answers['article_batch'] == 2]
batch3 = df_with_randomized_answers[df_with_randomized_answers['article_batch'] == 3]

In [33]:
from ftfy import fix_text

string_cols = [
    "paragraph",
    "question",
    "answer_A",
    "answer_B",
    "answer_C",
    "answer_D",
    "answer_1",
    "answer_2",
    "answer_3",
    'answer_0',
    'article_title',
    'rephrased_answer_A',
] 

for col in string_cols:
    batch3[col] = batch3[col].apply(
        lambda x: fix_text(x) if isinstance(x, str) else x
    )

In [34]:
batch1.to_csv(EXPERIMENT_DIR / 'batch1.tsv', index=False, encoding="utf-8", sep="\t")
batch2.to_csv(EXPERIMENT_DIR / 'batch2.tsv', index=False, encoding="utf-8", sep="\t")
batch3.to_csv(EXPERIMENT_DIR / 'batch3.tsv', index=False, encoding="utf-8", sep="\t")

In [40]:
batch1 = pd.read_csv(EXPERIMENT_DIR / 'batch1.tsv', encoding="utf-8", sep="\t")

In [41]:
batch1

,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,paragraph_id,...,answer_1,answer_2,answer_3,correct_answer,a_key,b_key,c_key,d_key,answers_order,is_experimental
0,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,"By 2050, animal-based protein consumption will...","By 2050, nine billion people will not have eno...",There will not be sufficient water to grow eno...,3,3,2,1,0,"[3, 2, 1, 0]",0
1,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,"By 2050, animal-based protein consumption will...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...",2,2,3,1,0,"[3, 2, 0, 1]",0
2,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,Obesity rates around the world will rise,"By 2050, animal-based protein consumption will...","By 2050, nine billion people will not have eno...",0,0,3,2,1,"[0, 3, 2, 1]",0
3,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,Obesity rates around the world will rise,There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...",2,2,3,0,1,"[2, 3, 0, 1]",0
4,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,Obesity rates around the world will rise,There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...",2,2,3,0,1,"[2, 3, 0, 1]",0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1075,There would probably be fewer accidents as dri...,"According to Kelley, what creates difficulties...",Waking up their teenage kids,Waking up before their teenage kids,Driving in the morning,Not having enough personal time,Getting their teenage children out of bed,2,9,6,...,Waking up their teenage kids,Waking up before their teenage kids,Not having enough personal time,1,1,2,0,3,"[2, 0, 1, 3]",1
1076,There would probably be fewer accidents as dri...,"According to Kelley, what creates difficulties...",Waking up their teenage kids,Waking up before their teenage kids,Driving in the morning,Not having enough personal time,Getting their teenage children out of bed,2,9,6,...,Waking up their teenage kids,Waking up before their teenage kids,Not having enough personal time,1,1,2,0,3,"[2, 0, 1, 3]",1
1077,There would probably be fewer accidents as dri...,"According to Kelley, what creates difficulties...",Waking up their teenage kids,Waking up before their teenage kids,Driving in the morning,Not having enough per

In [47]:
batch1.columns


Index(['paragraph', 'question', 'answer_A', 'answer_B', 'answer_C', 'answer_D',
       'rephrased_answer_A', 'onestopqa_question_id', 'article_id',
       'paragraph_id', 'article_batch', 'list_version', 'article_regime',
       'is_practice', 'question_list_version',
       'selected_onestopqa_question_id', 'final_list_id', 'list_num', 'batch',
       'article_title', 'answer_0', 'answer_1', 'answer_2', 'answer_3',
       'correct_answer', 'a_key', 'b_key', 'c_key', 'd_key', 'answers_order',
       'is_experimental'],
      dtype='str')

In [51]:
import pandas as pd

df = batch1.copy()

df = df.sort_values(
    ["list_num", "article_regime", "is_practice", "article_id", "paragraph_id"],
    ascending=[True, True, False, True, True]
)

df_2_paragraphs = (
    df.groupby(["list_num", "article_regime", "article_id"], group_keys=False)
      .head(2)
)


article_order = (
    df_2_paragraphs[["list_num", "article_regime", "article_id", "is_practice"]]
    .drop_duplicates()
    .sort_values(
        ["list_num", "article_regime", "is_practice", "article_id"],
        ascending=[True, True, False, True]
    )
)


article_order["article_keep_rank"] = (
    article_order
    .groupby(["list_num", "article_regime"])
    .cumcount()
)

articles_to_keep = article_order.loc[
    article_order["article_keep_rank"] < 3,
    ["list_num", "article_regime", "article_id"]
]

small_sample = df_2_paragraphs.merge(
    articles_to_keep,
    on=["list_num", "article_regime", "article_id"],
    how="inner"
)

small_sample.to_csv(EXPERIMENT_DIR / "small_sample.tsv", encoding="utf-8", sep="\t", index=False)

In [52]:
small_sample[(small_sample['final_list_id'] == 'B1_1a_Q0') & (small_sample['article_regime'] == 'no_knowledge')]

,paragraph,question,answer_A,answer_B,answer_C,answer_D,rephrased_answer_A,onestopqa_question_id,article_id,paragraph_id,...,answer_1,answer_2,answer_3,correct_answer,a_key,b_key,c_key,d_key,answers_order,is_experimental
6,Leading water scientists have issued one of th...,"According to Malik Falkenmark's report, what w...",There will not be sufficient water to grow eno...,"By 2050, nine billion people will not have eno...","By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,There won't be enough water to grow food for a...,0,0,1,...,"By 2050, animal-based protein consumption will...",Obesity rates around the world will rise,"By 2050, nine billion people will not have eno...",0,0,3,1,2,"[0, 2, 3, 1]",0
7,"""There will be just enough water if the propor...",Which factor led to the increase in the price ...,Unfavorable weather conditions in different lo...,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,Global warming,Bad weather in different locations on the planet,0,0,2,...,Global warming,Instability of the international financial mar...,Warnings of water scarcity by Oxfam and the UN,0,0,2,3,1,"[0, 3, 1, 2]",0
8,An octopus has made a brazen escape from the N...,How does Yarrell describe Inky's personality?,Curious,Nervous about what's happening on the outside,Unhappy,Playful and mischievous,Curious,2,7,1,...,Nervous about what's happening on the outside,Curious,Playful and mischievous,2,2,1,0,3,"[2, 1, 0, 3]",1
9,One theory is that Inky slid across the aquari...,How did Inky escape?,No one is sure - there are several theories th...,Through an open pipe at the top of his tank,Yarrell and other staff secretly helped him,Through a window that was accidentally left open,"The exact method is unknown, with several poss...",0,7,2,...,No one is sure - there are several theories th...,Through a window that was accidentally left open,Through an open pipe at the top of his tank,1,1,3,0,2,"[2, 0, 3, 1]",1
10,"Do you want your child to be good at sports, m...",What does the study say about the fitness of c...,They are weaker than children born in November...,"They are stronger than all children, except th...",They tend to be particularly good at sports,They tend to perform well on math and science ...,They are less fit than those born in November ...,0,8,1,...,They are weaker than children born in November...,"They are stronger than all children, except th...",They tend to be particularly good at sports,1,1,2,3,0,"[3, 0, 1, 2]",1
11,"The research involved 8,550 boys and girls age...",What were the study's conclusions concerning t...,November children ranked second in strength an...,November children scored highest in each category,November children came slightly behind Decembe...,November children did not differ from children...,"They placed first for stamina and power, and s...",1,8,2,...,November children came slightly behind Decembe...,November children ranked second in strength an...,November children did not differ from children...,2,2,0,1,3,"[1, 2, 0, 3]",1


In [5]:
res = pd.read_csv(EXPERIMENT_DIR / "fake result" / 'ia.tsv', encoding="utf-8", sep="\t")

In [9]:
import csv
import pandas as pd

msg = pd.read_csv(
    EXPERIMENT_DIR / "fake result" / "msg.tsv",
    encoding="utf-8",
    sep="\t",
    quoting=csv.QUOTE_NONE,
    engine="python"
)

In [ ]:
rep = pd.read_csv(
    EXPERIMENT_DIR / "fake result" / "rep.tsv",
    encoding="utf-8",
    sep="\t",
    quoting=csv.QUOTE_NONE,
    engine="python"
)

In [13]:
ia = pd.read_csv(
    EXPERIMENT_DIR / "fake result" / "ia.tsv",
    encoding="utf-8",
    sep="\t",
    quoting=csv.QUOTE_NONE,
    engine="python"
)

In [ ]:
ia

,RECORDING_SESSION_LABEL,TRIAL_INDEX,CURRENT_FIX_ADJUSTED,CURRENT_FIX_BLINK_AROUND,CURRENT_FIX_BUTTON_0_PRESS,CURRENT_FIX_BUTTON_1_PRESS,CURRENT_FIX_BUTTON_2_PRESS,CURRENT_FIX_BUTTON_3_PRESS,CURRENT_FIX_BUTTON_4_PRESS,CURRENT_FIX_BUTTON_5_PRESS,...,is_experimental,is_practice,list_num,list_version,onestopqa_question_id,paragraph_id,question,question_list_version,rephrased_answer_a,selected_onestopqa_question_id
0,Test2,1,False,NONE,.,.,.,.,.,.,...,0,True,0,1a,0,1,"According to Malik Falkenmark's report, what w...",0,There won't be enough water to grow food for a...,0
1,Test2,1,False,NONE,.,.,.,.,.,.,...,0,True,0,1a,0,1,"According to Malik Falkenmark's report, what w...",0,There won't be enough water to grow food for a...,0
2,Test2,1,False,NONE,.,.,.,.,.,.,...,0,True,0,1a,0,1,"According to Malik Falkenmark's report, what w...",0,There won't be enough water to grow food for a...,0
3,Test2,1,False,NONE,.,.,.,.,.,.,...,0,True,0,1a,0,1,"According to Malik Falkenmark's report, what w...",0,There won't be enough water to grow food for a...,0
4,Test2,1,False,NONE,.,.,.,.,.,.,...,0,True,0,1a,0,1,"According to Malik Falkenmark's report, what w...",0,There won't be enough water to grow food for a...,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2715,Test2,23,False,NONE,.,.,.,.,.,.,...,1,False,0,1a,0,2,What does Pullman find outrageous?,0,That people can steal artists' work without co...,0
2716,Test2,23,False,NONE,.,.,.,.,.,.,...,1,False,0,1a,0,2,What does Pullman find outrageous?,0,That people can steal artists' work without co...,0
2717,Test2,23,False,NONE,.,.,.,.,.,.,...,1,False,0,1a,0,2,What does Pullman find outrageous?,0,That people can steal artists' work without co...,0
2718,Test2,23,False,NONE,.,.,.,.,.,.,...,1,False,0,1a,0,2,What does Pullman find outrageous?,0,That people can steal artists' work without co...,0
